[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/17_tracing.ipynb)

# 17. Tracing

**Part 4: Reliability** · about 25 minutes

When an agent fails, the question is always *why*. Did the model misunderstand, did a tool return something confusing, did it run out of context, or did it just take a wrong turn at step 7? The printed trace we've used so far answers some of that, but it scrolls away, it's truncated, and you can't analyse it afterwards.

A **trace** is a structured record of everything an agent did: every model call and every tool call, with timings, token counts, inputs and outputs. Saved as a file, it lets you:

- see where the time and the tokens went
- find the exact step where a run went wrong
- compare two runs side by side
- and, the best part, **replay** a run later without calling the model at all

**By the end you will have:**

- a `Tracer` that writes one JSON line per event
- a text "waterfall" chart of a run
- a replay model that turns any recorded run into a free, repeatable test

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import difflib
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import tempfile
import time
import uuid
from pathlib import Path

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

## 1. The tracer

One event per line, in a format called **JSON Lines**: easy to append to while the run is going, easy to read back, and easy to load into any analysis tool. Each event has a type, a step number, a start time and a duration, plus whatever matters for that type.

We store the model's complete reply in each model-call event. That makes the trace bigger, but it's what makes replay possible in section 5.

In [3]:
class Tracer:
    def __init__(self, folder):
        self.run_id = uuid.uuid4().hex[:8]
        self.path = Path(folder) / f"run-{self.run_id}.jsonl"
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.started = time.perf_counter()

    def record(self, kind, step, start, **details):
        event = {"type": kind, "step": step, "start": round(start - self.started, 3),
                 "duration": round(time.perf_counter() - start, 3), **details}
        with self.path.open("a") as f:
            f.write(json.dumps(event) + "\n")
        return event


def load_trace(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines()]

## 2. An agent to trace

We use agent v2's task from notebook 10: a bug report about `wordcount.py` crashing on an empty file. The tools are trimmed to keep this notebook short. The only change to the loop is that every model call and tool call is wrapped in `tracer.record`.

In [4]:
WORDCOUNT = '''\
"""Print word statistics for a text file."""

import sys


def stats(text):
    words = text.split()
    total_letters = sum(len(word) for word in words)
    return {"words": len(words), "average_length": round(total_letters / len(words), 2)}


def main(path):
    with open(path) as f:
        result = stats(f.read())
    print(f"{result['words']} words, average length {result['average_length']}")


if __name__ == "__main__":
    main(sys.argv[1])
'''
TESTS = '''\
import unittest

from wordcount import stats


class StatsTests(unittest.TestCase):
    def test_counts_words(self):
        self.assertEqual(stats("one two three")["words"], 3)
'''


def make_workspace():
    folder = Path(tempfile.mkdtemp(prefix="agent-workspace-")).resolve()
    (folder / "wordcount.py").write_text(WORDCOUNT)
    (folder / "test_wordcount.py").write_text(TESTS)
    (folder / "empty.txt").write_text("")
    return folder


def make_tools(workspace):
    def safe(path):
        full = (workspace / path).resolve()
        if not full.is_relative_to(workspace):
            raise PermissionError(f"{path!r} is outside the workspace")
        return full

    def run_command(command):
        if not re.fullmatch(r"python3? (-m unittest[\w .-]*|[\w.-]+\.py[\w .-]*)", command.strip()):
            return "Denied: in this notebook only `python <script>` and `python -m unittest` are allowed."
        command = re.sub(r"^python3?", shlex.quote(sys.executable), command.strip())
        done = subprocess.run(command, shell=True, cwd=workspace, capture_output=True, text=True, timeout=30,
                              env={k: os.environ[k] for k in ("PATH", "SYSTEMROOT") if k in os.environ})
        output = (done.stdout + done.stderr).replace(str(workspace), ".")[-2000:]
        return f"exit code: {done.returncode}\n{output}"

    def read_file(path):
        return "\n".join(f"{n:>4} | {l}" for n, l in enumerate(safe(path).read_text().splitlines(), 1))

    def edit_file(path, old, new):
        text = safe(path).read_text()
        if text.count(old) != 1:
            return f"Error: old must appear exactly once; it appears {text.count(old)} times."
        safe(path).write_text(text.replace(old, new, 1))
        return f"Edited {path}."

    def create_file(path, content):
        if safe(path).exists():
            return f"Error: {path} already exists. Use edit_file to change it."
        safe(path).write_text(content)
        return f"Created {path}."

    return {"run_command": run_command, "read_file": read_file, "edit_file": edit_file, "create_file": create_file}


def schema(name, description, **props):
    return {"type": "function", "function": {"name": name, "description": description, "parameters": {
        "type": "object", "properties": {k: {"type": "string"} for k in props}, "required": list(props)}}}


TOOLS = [
    schema("run_command", "Run `python <script>` or `python -m unittest` in the project.", command=1),
    schema("read_file", "Read a file with line numbers.", path=1),
    schema("edit_file", "Replace text that appears exactly once in a file.", path=1, old=1, new=1),
    schema("create_file", "Create a new file. Fails if it exists.", path=1, content=1),
]
SYSTEM = ("You fix bugs. Reproduce the problem, read the code, fix it with edit_file, confirm the fix, "
          "add a test in a new file, run all tests with `python -m unittest`, then summarise briefly.")
TASK = "Bug report: `python wordcount.py empty.txt` crashes. Fix it and add a test."


def call_model(messages):
    payload = {"model": llm.model, "messages": messages, "tools": TOOLS}
    response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
    response.raise_for_status()
    return response.json()

In [5]:
def run_agent(tracer, model=call_model, workspace=None, max_steps=15):
    workspace = workspace or make_workspace()
    tools = make_tools(workspace)
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": TASK}]

    for step in range(1, max_steps + 1):
        start = time.perf_counter()
        data = model(messages)
        message = data["choices"][0]["message"]
        usage = data.get("usage") or {}
        tracer.record("model_call", step, start, input_tokens=usage.get("prompt_tokens", 0),
                      output_tokens=usage.get("completion_tokens", 0),
                      finish_reason=data["choices"][0].get("finish_reason"), message=message)
        messages.append(message)
        if not message.get("tool_calls"):
            return message.get("content"), workspace

        for call in message["tool_calls"]:
            start = time.perf_counter()
            name = call["function"]["name"]
            try:
                arguments = json.loads(call["function"]["arguments"] or "{}")
                result = tools[name](**arguments) if name in tools else f"Error: unknown tool {name!r}"
            except Exception as error:
                arguments, result = {}, f"Error: {type(error).__name__}: {error}"
            failed = result.startswith(("Error", "Denied")) or bool(re.match(r"exit code: [1-9]", result))
            tracer.record("tool_call", step, start, tool=name, arguments=arguments,
                          result=result[:2000], failed=failed)
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
    return None, workspace


traces = Path(tempfile.mkdtemp(prefix="agent-traces-"))
tracer = Tracer(traces)
answer, workspace = run_agent(tracer)
print("Answer:", answer)
print(f"\nTrace written: {tracer.path.name}, {len(load_trace(tracer.path))} events")

Answer: wordcount.py divided by the number of words, which crashes on an empty file. It now reports an average of 0.0 when there are no words. I added test_empty.py for this case, and all tests pass.

Trace written: run-235cff93.jsonl, 13 events


## 3. Reading a trace

First the raw material. Each line is one event:

In [6]:
events = load_trace(tracer.path)
for event in events[:3]:
    shown = {k: (v if k != "message" else "{...}") for k, v in event.items()}
    print(json.dumps(shown)[:180])

{"type": "model_call", "step": 1, "start": 0.001, "duration": 0.352, "input_tokens": 332, "output_tokens": 37, "finish_reason": "tool_calls", "message": "{...}"}
{"type": "tool_call", "step": 1, "start": 0.354, "duration": 0.068, "tool": "run_command", "arguments": {"command": "python wordcount.py empty.txt"}, "result": "exit code: 1\nTrace
{"type": "model_call", "step": 2, "start": 0.422, "duration": 0.347, "input_tokens": 516, "output_tokens": 31, "finish_reason": "tool_calls", "message": "{...}"}


Raw events are for machines. For humans, a **waterfall** is the most useful view: one row per event, positioned by when it started and sized by how long it took. Slow steps jump out, and so do patterns, like a model call that's always slower than the tools around it.

In [7]:
def waterfall(events, width=50):
    end = max(e["start"] + e["duration"] for e in events) or 1
    print(f"{'step':>4}  {'event':<28}{'time':>7}  timeline (total {end:.1f}s)")
    for e in events:
        label = "model" if e["type"] == "model_call" else f"  {e['tool']}"
        if e["type"] == "model_call" and e["message"].get("tool_calls"):
            label += " -> " + ", ".join(c["function"]["name"] for c in e["message"]["tool_calls"])
        offset = int(width * e["start"] / end)
        bar = max(1, int(width * e["duration"] / end))
        flag = "  FAILED" if e.get("failed") else ""
        print(f"{e['step']:>4}  {label[:28]:<28}{e['duration']:>6.2f}s  {' ' * offset}{'#' * bar}{flag}")


waterfall(events)

step  event                          time  timeline (total 3.3s)
   1  model -> run_command          0.35s  #####
   1    run_command                 0.07s       #  FAILED
   2  model -> read_file            0.35s        #####
   2    read_file                   0.00s             #
   3  model -> edit_file            0.35s             #####
   3    edit_file                   0.00s                   #
   4  model -> run_command          0.36s                   #####
   4    run_command                 0.06s                        #
   5  model -> create_file          0.35s                         #####
   5    create_file                 0.00s                              #
   6  model -> run_command          0.35s                              #####
   6    run_command                 0.09s                                    #
   7  model                         0.94s                                     ##############


"FAILED" on the first `run_command` is expected: that's the agent reproducing the crash. A run where the *last* `run_command` fails is the kind you'd want to dig into.

Now a summary, the numbers you'd put on a dashboard:

In [8]:
def summarise(events):
    model = [e for e in events if e["type"] == "model_call"]
    tools = [e for e in events if e["type"] == "tool_call"]
    total = max(e["start"] + e["duration"] for e in events)
    model_time, tool_time = sum(e["duration"] for e in model), sum(e["duration"] for e in tools)
    slowest = max(events, key=lambda e: e["duration"])
    return {
        "model calls": len(model),
        "tool calls": len(tools),
        "failed tool calls": sum(e["failed"] for e in tools),
        "input tokens": sum(e["input_tokens"] for e in model),
        "output tokens": sum(e["output_tokens"] for e in model),
        "time in model": f"{model_time:.1f}s ({100 * model_time / total:.0f}%)",
        "time in tools": f"{tool_time:.1f}s ({100 * tool_time / total:.0f}%)",
        "slowest event": f"step {slowest['step']} {slowest.get('tool', 'model call')}, {slowest['duration']:.2f}s",
    }


for key, value in summarise(events).items():
    print(f"{key:<20}{value}")

model calls         7
tool calls          6
failed tool calls   1
input tokens        5462
output tokens       359
time in model       3.0s (93%)
time in tools       0.2s (7%)
slowest event       step 7 model call, 0.94s


On a real model, "time in model" is usually 80 to 95 percent of the total. When we ran this on Gemma 4 E2B locally, it was over 99%: 296 seconds in six model calls against half a second in tools, with one model call alone taking 100 seconds. The same trace showed something else worth catching. The final answer was empty, even though the model had produced almost 3,900 output tokens over the run. Without a trace, all you'd see is a blank answer. That tells you where to optimise. A faster tool rarely matters. Fewer or shorter model calls almost always do. (Offline, the server's delays are simulated, so the split is less meaningful.)

## 4. Finding the step that went wrong

Traces make questions like "what did the model see right before it made that bad edit?" answerable. Here's a small helper that shows any step in full: what the model decided, and what each tool returned.

In [9]:
def show_step(events, step):
    for e in events:
        if e["step"] != step:
            continue
        if e["type"] == "model_call":
            message = e["message"]
            if message.get("content"):
                print(f"model said: {message['content'][:300]}")
            for call in message.get("tool_calls") or []:
                print(f"model called: {call['function']['name']}({call['function']['arguments'][:200]})")
        else:
            print(f"{e['tool']} returned{' (FAILED)' if e['failed'] else ''}:\n{e['result'][:400]}")


first_edit = next(e["step"] for e in events if e.get("tool") == "edit_file")
print(f"--- step {first_edit}, the first edit ---")
show_step(events, first_edit)

--- step 3, the first edit ---
model called: edit_file({"path": "wordcount.py", "old": "round(total_letters / len(words), 2)", "new": "round(total_letters / len(words), 2) if words else 0.0"})
edit_file returned:
Edited wordcount.py.


## 5. Replay: a recorded run becomes a free test

This is the trick that makes traces worth the disk space. The trace holds every reply the model gave. So we can build a **fake model that gives the same replies in the same order**, and run the agent again with no API calls, no cost and no randomness.

Why bother? Because the *model* isn't the only thing that changes. Your tools, your sandbox and your prompts all change too. Replaying a known-good run after you change a tool tells you at once whether the tool still behaves the way the model expected. If the replayed agent asks the tool for something and gets a different answer than last time, you've found a regression.

In [10]:
class ReplayModel:
    """Answers with the replies recorded in a trace, in order, and checks the agent stays on script."""

    def __init__(self, events):
        self.replies = [e["message"] for e in events if e["type"] == "model_call"]
        self.calls = 0

    def __call__(self, messages):
        if self.calls >= len(self.replies):
            raise RuntimeError("Replay diverged: the agent asked for more steps than the recording has.")
        reply = self.replies[self.calls]
        self.calls += 1
        return {"choices": [{"message": reply, "finish_reason": "tool_calls" if reply.get("tool_calls") else "stop"}],
                "usage": {}}


replay_tracer = Tracer(traces)
replayed_answer, replay_workspace = run_agent(replay_tracer, model=ReplayModel(events))

original = [(e["tool"], e["result"]) for e in events if e["type"] == "tool_call"]
replayed = [(e["tool"], e["result"]) for e in load_trace(replay_tracer.path) if e["type"] == "tool_call"]
print("Replayed answer matches:", replayed_answer == answer)
print("Every tool returned the same result:", original == replayed)

Replayed answer matches: True
Every tool returned the same result: True


In [11]:
# Checkpoint: a replay of an unchanged system reproduces the run exactly.
assert replayed_answer == answer and original == replayed
print("Checkpoint passed.")

Checkpoint passed.


Now let's break something on purpose, the way a careless refactor would. Suppose someone "improves" `edit_file` so it strips trailing whitespace from the file. Will the recorded run still work?

In [12]:
original_make_tools = make_tools


def make_tools(workspace):
    tools = original_make_tools(workspace)
    edit = tools["edit_file"]

    def edit_file(path, old, new):  # the "improvement": also strips trailing spaces from every line
        result = edit(path, old, new)
        file = workspace / path
        file.write_text("\n".join(line.rstrip() for line in file.read_text().splitlines()))
        return result.replace("Edited", "Edited and cleaned")

    tools["edit_file"] = edit_file
    return tools


changed_tracer = Tracer(traces)
run_agent(changed_tracer, model=ReplayModel(events))
changed = [(e["tool"], e["result"]) for e in load_trace(changed_tracer.path) if e["type"] == "tool_call"]

for (tool, before), (_, after) in zip(original, changed):
    if before != after:
        print(f"{tool} now returns something different:")
        for line in difflib.unified_diff(before.splitlines(), after.splitlines(), "recorded", "now", lineterm="", n=0):
            print("  " + line)
        break

make_tools = original_make_tools

edit_file now returns something different:
  --- recorded
  +++ now
  @@ -1 +1 @@
  -Edited wordcount.py.
  +Edited and cleaned wordcount.py.


The replay caught it straight away, without a model call. Here the change was harmless enough: a different message. But the same check would catch a tool that started returning errors, different line numbers or a changed output format, all things a model can trip over. Teams that build agents keep a folder of recorded traces from good runs and replay them all on every change, just like a test suite.

One limit: a replay only checks the path the model took **last time**. If a change would make a real model behave differently, replay can't know. That's what evals are for, in the next notebook.

## Recap

- A trace is a structured record of every model and tool call, saved as JSON Lines.
- A waterfall shows where the time goes. Model calls usually dominate.
- `show_step` answers "what did the model see when it made that choice?"
- Recording full model replies lets you **replay** runs as free, deterministic regression tests.
- Production systems often use OpenTelemetry for this. The ideas are exactly the same.

## Exercises

1. Add the context size from notebook 11 to every model-call event, and plot it over the steps.
2. Write `compare(trace_a, trace_b)`, which prints the first step where two runs of the same task made different tool calls.
3. Record a run on a real model, then replay it in offline mode. What breaks, and why?

## Clean up

In [13]:
for folder in [traces, workspace, replay_workspace]:
    shutil.rmtree(folder, ignore_errors=True)
print("Removed the temporary folders.")

Removed the temporary folders.


**Next:** [18. Evals](18_evals.ipynb). Measuring whether your agent is any good, and whether a change made it better.